# Qwen3.5 DataMate — benchmark trên tập train (Kaggle T4 ×2)
Bật Internet, GPU T4 ×2 và gắn dataset. Chỉ dùng tập **train** (có `answers.jsonl` để chấm).
Chạy `LIMIT = 1` trước để kiểm tra model và VRAM, rồi tăng `LIMIT` (0 = toàn bộ).

In [ ]:
REPO_URL = "https://github.com/KhoaNotFound/ZAIC2026-BackpropEverytTime.git"
DATA_DIR = ""  # để trống: tự tìm thư mục train trong /kaggle/input; hoặc dán đường dẫn cụ thể
LIMIT = 1  # số câu chạy thử; 0 = toàn bộ
SEED = 42
CONTEXT_TOKENS = 32768  # input + output mỗi bước; rulebook + README đã ~12-13k token nên 16384 là quá nhỏ
RESUME = False

import os
from pathlib import Path

def find_train_dirs(root="/kaggle/input"):
    """Thư mục chứa đủ questions.jsonl, answers.jsonl, rulebook.md và data/."""
    found = []
    for d, subdirs, files in os.walk(root):
        if {"questions.jsonl", "answers.jsonl", "rulebook.md"} <= set(files) and "data" in subdirs:
            found.append(d)
            subdirs.clear()
    return sorted(found)

if not DATA_DIR:
    candidates = find_train_dirs()
    print("Thư mục train tìm thấy:", candidates)
    assert len(candidates) == 1, "Không tìm thấy hoặc có nhiều hơn một thư mục train: gán DATA_DIR bằng một đường dẫn ở trên"
    DATA_DIR = candidates[0]
print("DATA_DIR =", DATA_DIR)
OUTPUT = f"/kaggle/working/qwen_train_limit{LIMIT}_seed{SEED}_ctx{CONTEXT_TOKENS}/predictions.jsonl"  # đổi cấu hình -> thư mục mới
print("OUTPUT =", OUTPUT)

In [ ]:
import subprocess
from pathlib import Path
repo = Path("/kaggle/working/zAIC")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
%cd /kaggle/working/zAIC
%pip install -r requirements.txt


Nếu pip yêu cầu restart kernel, restart rồi chạy lại cell cấu hình trước khi tiếp tục.


In [ ]:
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, round(props.total_memory / 2**30, 2), "GiB")


In [ ]:
import subprocess, sys, os
os.chdir("/kaggle/working/zAIC")
from pathlib import Path
assert (Path(DATA_DIR) / "questions.jsonl").is_file(), "Sửa DATA_DIR"
assert (Path(DATA_DIR) / "answers.jsonl").is_file(), "DATA_DIR phải là tập train có answers.jsonl"
base = [sys.executable, "inference.py", "--data-dir", DATA_DIR, "--limit", str(LIMIT), "--seed", str(SEED), "--output", OUTPUT]
subprocess.run(base + ["--dry-run"], check=True)


In [ ]:
import torch
assert torch.cuda.device_count() >= 2, "Cần 2 GPU: chọn Accelerator GPU T4 x2"
command = base + ["--model", "Qwen/Qwen3.5-9B", "--quantization", "4bit", "--device-map", "split",
                   "--context-tokens", str(CONTEXT_TOKENS)]
if RESUME:
    command += ["--resume"]
subprocess.run(command, check=True)


In [ ]:
subprocess.run([sys.executable, "evaluate.py", "--predictions", OUTPUT,
                "--answers", str(Path(DATA_DIR)/"answers.jsonl"),
                "--questions", str(Path(DATA_DIR)/"questions.jsonl")], check=True)
# Benchmark đầy đủ (theo loại đáp án, cửa hàng, thời gian, token, VRAM): predictions.evaluation.json

In [ ]:
summary = Path(OUTPUT).with_suffix(".summary.json")
if summary.exists():
    print(summary.read_text())
print("Artifacts:", [p.name for p in Path(OUTPUT).parent.glob("predictions.*")])


Đổi số câu: sửa `LIMIT` ở cell cấu hình, chạy lại từ đó (thư mục output tự đổi theo LIMIT/SEED). Chạy tiếp một đợt bị ngắt: giữ nguyên cấu hình, đặt `RESUME = True`. Save Version để giữ output.

Nếu chạy bị dừng do lỗi (ví dụ OOM), xem `predictions.traces.jsonl`; restart kernel trước khi chạy lại với output mới.